# 데이터 불러오기

In [48]:
import pandas as pd

# 저장했던 CSV 파일 1개만 읽어오기
df = pd.read_csv('./processed_data/combined_port_data.csv')

print("데이터 로드 완료! 크기:", df.shape)
df.head()

데이터 로드 완료! 크기: (103365, 22)


,순번,호출부호,선박명,입항횟수_연도,입항횟수_횟수,시설사용횟수,신고톤수,선사/대리점_코드,선사/대리점명,요청시설_코드,...,요청일시_FROM,요청일시_TO,지정시설_코드,지정시설_번호,지정시설명,지정일시_FROM,지정일시_TO,사용목적명,신청일시,허가유무
0,1,215802,장군2호,2024,1,1,39.0,ICZ3733,혜성,*JG,...,2024-01-30 17:40,2024-01-31 07:30,*JG,01,자갈치,2024-01-30 17:40,2024-01-31 07:30,출항대기,2024-01-31 22:34,허가완료
1,2,UBCZ2,MERIDA,2024,1,1,5.275,BSC2144,글로벌오프쇼어마린서비스(주),WAN,...,2024-02-01 10:00,2024-02-01 19:00,WAN,03,남외항 N-3박지,2024-02-01 10:00,2024-02-01 19:00,급유,2024-01-31 22:29,허가완료
2,3,3WON,SEA DREAM,2024,1,1,8.216,BSC2131,코리아머천트쉬핑(주),WAN,...,2024-02-01 05:00,2024-02-01 15:00,WAN,03,남외항 N-3박지,2024-02-01 05:00,2024-02-01 15:00,급유,2024-01-31 22:28,허가완료
3,4,D7XB,PANCON BRIDGE,2024,2,2,18.04,KMC8410,범주해운(주),WAN,...,2024-02-01 04:00,2024-02-02 13:00,MBS,05,신선대부두 5선석,2024-02-01 04:00,2024-02-02 13:00,접안대기,2024-01-31 22:28,허가완료
4,5,DSRL4,KMTC SHANGHAI,2024,2,1,20.815,KMS8832,고려해운(주),WAN,...,2024-02-01 00:01,2024-02-02 10:00,WAN,04,남외항 N-4박지,2024-02-01 00:01,2024-02-02 10:00,접안대기,2024-01-31 22:26,허가완료


3.1 컬럼 제거 및 결측치 확인

In [24]:
# 2. 불필요한 컬럼 제거
# - 대기시간 분석 및 접안 정체 분석에 필요 없는 컬럼들 정리
columns_to_drop = [
    '순번', '시설사용횟수',
    '선사대리점_코드', '선사대리점명',
    '요청시설_코드', '요청시설_번호',
    '지정시설_코드', '지정시설_번호',
    '부선호출부호', '부선명', '부선호출부호2', '부선명2',
    '외내', 'CIQ수속일자', 'MRN 번호', '차항지', '전출항지',
    '외항:한국인선원수/내항:해기사선원수', '외항:외국인선원수/내항:보통선원수',
    '승객', '예선', '도선', '부선호출부호1'
]

# errors='ignore'로 존재하지 않는 컬럼 오류 방지
df.drop(columns=columns_to_drop, errors='ignore', inplace=True)
print("불필요한 컬럼 삭제 완료!")

불필요한 컬럼 삭제 완료!


In [25]:
# 3. 데이터 조건 필터링
# '허가유무'가 '미허가'가 아닌 승인/허가 건만 남김
if '허가유무' in df.columns:
    df = df[df['허가유무'] != '미허가']

# '입출' 컬럼이 존재하는 경우 '출항' 조건 적용
if '입출' in df.columns:
    df = df[df['입출'] == '출항']

3.2 데이터 병합(Merge)

In [26]:
# 4. 수치형 컬럼 형변환 (숫자형으로 변환 및 float 타입 적용)
# pd.to_numeric(..., errors='coerce')는 문자열이나 결측치를 안전하게 숫자로 바꿔줍니다.
if '입항횟수_연도' in df.columns:
    df['입항횟수_연도'] = pd.to_numeric(df['입항횟수_연도'], errors='coerce').astype('Int64')

if '입항횟수_횟수' in df.columns:
    df['입항횟수_횟수'] = pd.to_numeric(df['입항횟수_횟수'], errors='coerce').astype('Int64')

if '신고톤수' in df.columns:
    df['신고톤수'] = pd.to_numeric(df['신고톤수'], errors='coerce').astype(float)

print("정제, 필터링 및 숫자 형변환 완료!")
print("최종 데이터 크기:", df.shape)
print("\n[현재 남은 컬럼 목록]:")
print(list(df.columns))

df.head()

정제, 필터링 및 숫자 형변환 완료!
최종 데이터 크기: (90128, 16)

[현재 남은 컬럼 목록]:
['호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수', '신고톤수', '선사/대리점_코드', '선사/대리점명', '요청시설명', '요청일시_FROM', '요청일시_TO', '지정시설명', '지정일시_FROM', '지정일시_TO', '사용목적명', '신청일시', '허가유무']


,호출부호,선박명,입항횟수_연도,입항횟수_횟수,신고톤수,선사/대리점_코드,선사/대리점명,요청시설명,요청일시_FROM,요청일시_TO,지정시설명,지정일시_FROM,지정일시_TO,사용목적명,신청일시,허가유무
0,215802,장군2호,2024,1,39.000,ICZ3733,혜성,자갈치,2024-01-30 17:40,2024-01-31 07:30,자갈치,2024-01-30 17:40,2024-01-31 07:30,출항대기,2024-01-31 22:34,허가완료
1,UBCZ2,MERIDA,2024,1,5.275,BSC2144,글로벌오프쇼어마린서비스(주),남외항 N-3박지,2024-02-01 10:00,2024-02-01 19:00,남외항 N-3박지,2024-02-01 10:00,2024-02-01 19:00,급유,2024-01-31 22:29,허가완료
2,3WON,SEA DREAM,2024,1,8.216,BSC2131,코리아머천트쉬핑(주),남외항 N-3박지,2024-02-01 05:00,2024-02-01 15:00,남외항 N-3박지,2024-02-01 05:00,2024-02-01 15:00,급유,2024-01-31 22:28,허가완료
3,D7XB,PANCON BRIDGE,2024,2,18.040,KMC8410,범주해운(주),남외항 N-4박지,2024-02-01 04:00,2024-02-02 13:00,신선대부두 5선석,2024-02-01 04:00,2024-02-02 13:00,접안대기,2024-01-31 22:28,허가완료
4,DSRL4,KMTC SHANGHAI,2024,2,20.815,KMS8832,고려해운(주),남외항 N-4박지,2024-02-01 00:01,2024-02-02 10:00,남외항 N-4박지,2024-02-01 00:01,2024-02-02 10:00,접안대기,2024-01-31 22:26,허가완료


3.3 대기시간, 서비스 시간 산출

In [27]:
# 중복된 행, unique값만 남기기
unique_values = df.drop_duplicates()

In [28]:
temp = unique_values.copy()

temp['지정일시_TO'] = pd.to_datetime(temp['지정일시_TO'], errors='coerce')
temp['지정일시_FROM'] = pd.to_datetime(temp['지정일시_FROM'], errors='coerce')

duration = temp['지정일시_TO'] - temp['지정일시_FROM']

temp['접안_대기시간'] = duration.where(
    temp['사용목적명'] == '접안대기'
)

temp['출항_대기시간'] = duration.where(
    temp['사용목적명'] == '출항대기'
)

service_Time_End 생성
- 출항대기시간 null 값인 경우에는, Service_Time_End 값이 '출항일시' 값
- 사용목적명이 '출항대기'일 경우에 Service_Time_End 값이 '지정일시(FROM)' 값

In [30]:
temp['Service_Time_End'] = pd.NaT

# 출항대기인 경우 시작시각을 Service_Time_End로
temp.loc[
    temp['사용목적명'] == '출항대기',
    'Service_Time_End'
] = temp['지정일시_FROM']

# 출항_대기시간이 null인 경우 지정일시_TO 사용
temp.loc[
    temp['출항_대기시간'].isnull(),
    'Service_Time_End'
] = temp['지정일시_TO']

AssertionError: Something has gone wrong, please report a bug at https://github.com/pandas-dev/pandas/issues

- Anchor(=접안대기), Service_Time_Start 생성

In [38]:
temp['Anchor'] = np.nan

# '사용목적명'이 '접안대기'인 경우 'Anchor' 값 할당
temp.loc[temp['사용목적명'] == '접안대기', 'Anchor'] = temp['지정일시(TO)']

# Anchor(=접안대기)가 null 인 경우, 입항 일시 값을 service time start로
temp['Service_Time_Start'] = np.where(temp['Anchor'].isnull(), temp['입항일시'], temp['Anchor'])

# 문자열을 datetime으로 변환
temp['Service_Time_End'] = pd.to_datetime(temp['Service_Time_End'])
temp['Service_Time_Start'] = pd.to_datetime(temp['Service_Time_Start'])
temp['Service_Time'] = temp['Service_Time_End'] - temp['Service_Time_Start']

KeyError: '지정일시(TO)'

대기율 산출

In [40]:
waiting = temp.copy()

# 접안_대기시간이 null 값인 경우 대기 0
waiting['접안_대기시간'].fillna(0, inplace = True)

# 데이터프레임에서 '접안_대기시간'과 'Service_Time' 컬럼을 timedelta 형식으로 변환
waiting['접안_대기시간'] = pd.to_timedelta(waiting['접안_대기시간'])
waiting['Service_Time'] = pd.to_timedelta(waiting['Service_Time'])

# timedelta 값에서 일(day)로 변환
waiting['접안_대기시간_NumDays'] = waiting['접안_대기시간'].dt.days
waiting['Service_Time_NumDays'] = waiting['Service_Time'].dt.days

# 'Service_Time' 컬럼에서 'days', 'hours', 'minutes'를 분리하여 새로운 열로 추가
waiting['Service_Time_일'] = waiting['Service_Time'].dt.days
waiting['Service_Time_시간'] = waiting['Service_Time'].dt.components.hours
waiting['Service_Time_분'] = waiting['Service_Time'].dt.components.minutes

# 'Service_Time'을 분으로 변환하여 'Service_Time_분' 열에 추가
waiting['Service_Time_분'] = waiting['Service_Time_일'] * 24 * 60 + waiting['Service_Time_시간'] * 60 + waiting['Service_Time_분']

# '접안_대기시간'을 분으로 변환하여 '접안_대기시간_분' 열에 추가
waiting['접안_대기시간_분'] = waiting['접안_대기시간_일'] * 24 * 60 + waiting['접안_대기시간_시간'] * 60 + waiting['접안_대기시간_분']

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_32824\3436518222.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  waiting['접안_대기시간'].fillna(0, inplace = True)


TypeError: Invalid value '0' for dtype 'datetime64[ns]'

In [41]:
waiting['대기율'] = waiting['접안_대기시간_분'] / waiting['Service_Time_분']

KeyError: '접안_대기시간_분'

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# 1. 저장한 CSV 불러오기
df = pd.read_csv('./processed_data/merged_df.csv')

df

,항명,호출부호,선박명,입항횟수_연도,입항횟수_횟수,구분,입출,총톤수,입항일시,출항일시,...,계선장소명,선박용도,신고톤수,지정시설_코드,지정시설명,지정일시(FROM),지정일시(TO),사용목적명,신청일시,허가유무
0,부산,796256,백경3,2004.0,1.0,최종,출항,22,2004-01-01 00:00,2024-10-23 08:00,...,기타 남항 사설조선소,석유제품 운반선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,부산,058251,대은호,2016.0,2.0,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,5물량장 1선석,급유선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,부산,915291,17금영호,2019.0,1.0,최종,출항,22,2019-05-30 09:00,2024-08-29 15:00,...,감천 선기조합조선소 안벽-02,급유선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,부산,JVNF8,SUNRISE,2019.0,1.0,최종,출항,19,2019-11-21 18:00,2024-05-10 11:30,...,수영박지(집단박지),수상레저기구,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,부산,030713,청운호,2021.0,1.0,최초,출항,121,2021-01-01 00:00,2024-09-30 00:00,...,5물량장 1선석,석유제품 운반선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45135,부산,3FBU8,ROSEMARY,2024.0,1.0,최종,출항,"93,526",2024-12-31 17:30,2024-12-31 21:00,...,남외항 N-5박지,산물선(벌크선),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45136,부산,D5NP9,SAEHAN VIRTUS,2024.0,10.0,최종,출항,"3,428",2024-12-31 17:36,2024-12-31 20:00,...,남외항 N-3박지,LPG 운반선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45137,부산,060333,15영신,2024.0,728.0,최종,출항,40,2024-12-31 17:40,2024-12-31 18:20,...,남외항 N-5박지,기타선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45138,부산,D8AA,울산709호,2024.0,29.0,최종,출항,130.98,2024-12-31 20:45,2024-12-31 21:00,...,감천 4부두 5선석,기타 예선,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# 2. 날짜/시간 컬럼 datetime 형식 변환
datetime_cols = ['지정일시(FROM)', '지정일시(TO)', '입항일시', '출항일시']
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

In [22]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# 1. 저장했던 merged_df CSV 파일 불러오기
df = pd.read_csv('./processed_data/merged_df.csv')
print("데이터 로드 완료! 크기:", df.shape)

# 2. 날짜/시간 컬럼 datetime 형식 변환
datetime_cols = ['지정일시(FROM)', '지정일시(TO)', '입항일시', '출항일시']
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. 지표(서비스시간, 대기시간, 대기율) 계산 진행
temp = df.copy()

# ★ [핵심 해결] 조건 판별 후 타입 꼬임 없이 바로 Timedelta 계산
# 접안대기시간: 접안대기인 행만 계산, 아닌 경우 NaT
mask_접안 = temp['사용목적명'] == '접안대기'
temp['접안_대기시간'] = pd.Series(pd.NaT, index=temp.index, dtype='timedelta64[ns]')
temp.loc[mask_접안, '접안_대기시간'] = temp.loc[mask_접안, '지정일시(TO)'] - temp.loc[mask_접안, '지정일시(FROM)']

# 출항대기시간: 출항대기인 행만 계산, 아닌 경우 NaT
mask_출항 = temp['사용목적명'] == '출항대기'
temp['출항_대기시간'] = pd.Series(pd.NaT, index=temp.index, dtype='timedelta64[ns]')
temp.loc[mask_출항, '출항_대기시간'] = temp.loc[mask_출항, '지정일시(TO)'] - temp.loc[mask_출항, '지정일시(FROM)']


# Service_Time_End 산출
temp['Service_Time_End'] = temp['출항일시']
temp.loc[mask_출항, 'Service_Time_End'] = temp.loc[mask_출항, '지정일시(FROM)']

# Anchor 및 Service_Time_Start 산출
temp['Anchor'] = pd.NaT
temp.loc[mask_접안, 'Anchor'] = temp.loc[mask_접안, '지정일시(TO)']

# Anchor가 있으면 Anchor, 없으면 입항일시 적용
temp['Service_Time_Start'] = temp['Anchor'].fillna(temp['입항일시'])

# Service_Time 계산
temp['Service_Time'] = temp['Service_Time_End'] - temp['Service_Time_Start']


# 4. 대기율 산출
waiting = temp.copy()

# 접안대기시간 결측치는 0초로 대체
waiting['접안_대기시간'] = waiting['접안_대기시간'].fillna(pd.Timedelta(seconds=0))

# Timedelta를 분(Minutes) 단위로 변환
waiting['Service_Time_분'] = waiting['Service_Time'].dt.total_seconds() / 60
waiting['접안_대기시간_분'] = waiting['접안_대기시간'].dt.total_seconds() / 60

# 대기율 = 접안대기시간(분) / 서비스시간(분)
waiting['대기율'] = waiting['접안_대기시간_분'] / waiting['Service_Time_분']

# 이상치 처리 (서비스시간이 0 이하이거나 무한대인 값 제외)
valid_waiting = waiting[
    (waiting['Service_Time_분'] > 0) &
    (~waiting['대기율'].isin([np.inf, -np.inf]))
]

print(f"2024년 평균 대기율: {valid_waiting['대기율'].mean():.6f}")

데이터 로드 완료! 크기: (45140, 24)
2024년 평균 대기율: 0.000000


In [ ]:
# 4. 대기율 산출
waiting = temp.copy()

# 접안대기시간 결측치는 0초 Timedelta로 대체
waiting['접안_대기시간'] = waiting['접안_대기시간'].fillna(pd.Timedelta(seconds=0))

# Timedelta를 분(Minutes) 단위로 변환
waiting['Service_Time_분'] = waiting['Service_Time'].dt.total_seconds() / 60
waiting['접안_대기시간_분'] = waiting['접안_대기시간'].dt.total_seconds() / 60

# 대기율 = 접안대기시간(분) / 서비스시간(분)
waiting['대기율'] = waiting['접안_대기시간_분'] / waiting['Service_Time_분']

# 이상치 처리 (서비스시간이 0 이하이거나 무한대인 값 제외)
valid_waiting = waiting[
    (waiting['Service_Time_분'] > 0) &
    (~waiting['대기율'].isin([np.inf, -np.inf]))
]

print(f"2024년 평균 대기율: {valid_waiting['대기율'].mean():.6f}")

In [32]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 저장한 병합 데이터 불러오기
# ---------------------------------------------------------
df = pd.read_csv('./data/merged_facility_ship.csv')

# 컬럼명 좌우 공백 제거 (에러 방지)
df.columns = df.columns.str.strip()

# ---------------------------------------------------------
# 2. 날짜 컬럼 자동 감지 및 Datetime 변환
# ---------------------------------------------------------
# 후보가 될 수 있는 컬럼명들 확인
from_col = next((c for c in ['지정일시(FROM)', '지정일시', '지정기간(FROM)'] if c in df.columns), None)
app_col = next((c for c in ['신청일시', '요청일시'] if c in df.columns), None)

print(f"감지된 시작시간 컬럼: {from_col}")
print(f"감지된 신청시간 컬럼: {app_col}")

# Datetime 형변환
if from_col:
    df[from_col] = pd.to_datetime(df[from_col], errors='coerce')
if app_col:
    df[app_col] = pd.to_datetime(df[app_col], errors='coerce')

# ---------------------------------------------------------
# 3. 대기시간(시간) 산정
# ---------------------------------------------------------
if from_col and app_col:
    # 대기시간 = 지정일시 - 신청일시 (시간 단위)
    df['대기시간(시간)'] = (df[from_col] - df[app_col]).dt.total_seconds() / 3600.0

    # 음수값(신청 전 지정 등 오류 데이터) 0으로 보정
    df['대기시간(시간)'] = df['대기시간(시간)'].apply(lambda x: max(x, 0) if pd.notnull(x) else np.nan)
else:
    # 지정일시/신청일시 컬럼이 없을 경우 대체 계산값 생성
    print("⚠️ 지정일시 또는 신청일시 컬럼을 찾지 못해 대기시간을 0으로 초기화합니다.")
    df['대기시간(시간)'] = 0.0

# ---------------------------------------------------------
# 4. 사용목적별 / 지정시설별 대기시간 집계
# ---------------------------------------------------------
# 1) 사용목적별 대기시간 집계
if '사용목적명' in df.columns:
    purpose_stats = df.groupby('사용목적명')['대기시간(시간)'].agg(
        요청건수='count',
        평균대기시간_시간='mean',
        최대대기시간_시간='max'
    ).reset_index()
    print("\n=== 사용목적별 대기시간 집계 ===")
    print(purpose_stats)

# 2) 지정시설별 대기시간 집계
facility_col = next((c for c in ['지정시설명', '지정시설'] if c in df.columns), None)

if facility_col:
    facility_stats = df.groupby(facility_col)['대기시간(시간)'].agg(
        요청건수='count',
        평균대기시간_시간='mean',
        최대대기시간_시간='max'
    ).reset_index()
    print(f"\n=== {facility_col}별 대기시간 집계 ===")
    print(facility_stats.head())

감지된 시작시간 컬럼: None
감지된 신청시간 컬럼: 신청일시
⚠️ 지정일시 또는 신청일시 컬럼을 찾지 못해 대기시간을 0으로 초기화합니다.

=== 사용목적별 대기시간 집계 ===
     사용목적명   요청건수  평균대기시간_시간  최대대기시간_시간
0     갑문통과      5        0.0        0.0
1       계선   1251        0.0        0.0
2       급유   4749        0.0        0.0
3       기타  20866        0.0        0.0
4     상가대기    202        0.0        0.0
5       수리   2439        0.0        0.0
6   수입화물장치      2        0.0        0.0
7      양적하  15112        0.0        0.0
8       양하  11116        0.0        0.0
9       적하   5848        0.0        0.0
10    접안대기  13663        0.0        0.0
11    출항대기   6159        0.0        0.0
12    통과선박    171        0.0        0.0
13      해난      7        0.0        0.0
14    해상하역     31        0.0        0.0
15    화물대기    493        0.0        0.0

=== 지정시설별 대기시간 집계 ===
  지정시설  요청건수  평균대기시간_시간  최대대기시간_시간
0  *AG    41        0.0        0.0
1  *DL    64        0.0        0.0
2  *DP    21        0.0        0.0
3  *DS    34        0.0        0.0
4  *GY    17       

In [39]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 저장한 병합 데이터 불러오기
# ---------------------------------------------------------
df = pd.read_csv('./data/merged_facility_ship.csv')

# 컬럼명 내 줄바꿈(\n), 공백 정제 (핵심!)
df.columns = [str(c).replace('\n', '').replace(' ', '') for c in df.columns]

# 정제된 컬럼 목록 확인 출력
print("현재 데이터프레임 컬럼 목록:")
print(list(df.columns))

# ---------------------------------------------------------
# 2. 지정일시(FROM) 및 신청일시 컬럼 유연 감지
# ---------------------------------------------------------
from_col = next((c for c in df.columns if '지정일시' in c and 'FROM' in c.upper()), None)
if not from_col:
    # 지정일시가 한 줄로 들어간 경우 대체
    from_col = next((c for c in df.columns if '지정' in c and ('시작' in c or 'FROM' in c.upper() or '일시' in c)), None)

app_col = next((c for c in df.columns if '신청' in c or '요청' in c), None)

print(f"\n매칭된 지정시작 컬럼: {from_col}")
print(f"매칭된 신청일시 컬럼: {app_col}")

# ---------------------------------------------------------
# 3. 대기시간 산정
# ---------------------------------------------------------
if from_col and app_col:
    df[from_col] = pd.to_datetime(df[from_col], errors='coerce')
    df[app_col] = pd.to_datetime(df[app_col], errors='coerce')

    # 대기시간(시간) = 지정시작 - 신청일시
    df['대기시간(시간)'] = (df[from_col] - df[app_col]).dt.total_seconds() / 3600.0
    df['대기시간(시간)'] = df['대기시간(시간)'].apply(lambda x: max(x, 0) if pd.notnull(x) else np.nan)

    print("\n✅ 대기시간 계산 완료!")
else:
    print("\n⚠️ 지정시작 또는 신청일시 컬럼을 매칭하지 못했습니다. 상단 컬럼 목록을 확인해 주세요.")

# ---------------------------------------------------------
# 4. 집계 수행
# ---------------------------------------------------------
if '대기시간(시간)' in df.columns:
    purpose_col = next((c for c in df.columns if '사용목적' in c), None)
    facility_col = next((c for c in df.columns if '지정시설' in c and '명' in c), None)

    if purpose_col:
        print(f"\n=== {purpose_col}별 대기시간 집계 ===")
        print(df.groupby(purpose_col)['대기시간(시간)'].agg(요청건수='count', 평균대기시간_시간='mean', 최대대기시간_시간='max').reset_index())

    if facility_col:
        print(f"\n=== {facility_col}별 대기시간 집계 (상위 10개) ===")
        print(df.groupby(facility_col)['대기시간(시간)'].agg(요청건수='count', 평균대기시간_시간='mean', 최대대기시간_시간='max').reset_index().head(10))

현재 데이터프레임 컬럼 목록:
['항명', '호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수', '구분', '입출', '총톤수', '입항일시', '출항일시', '수리일시', '항해구분', '계선장소_코드', '계선장소_숫자', '계선장소명', '선박용도', '신고톤수', '지정시설', '사용목적명', '신청일시', '허가유무']

매칭된 지정시작 컬럼: None
매칭된 신청일시 컬럼: 신청일시

⚠️ 지정시작 또는 신청일시 컬럼을 매칭하지 못했습니다. 상단 컬럼 목록을 확인해 주세요.


In [47]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 병합 데이터 불러오기 및 컬럼명 완전 정제
# ---------------------------------------------------------
df = pd.read_csv('./data/merged_facility_ship.csv')

# 컬럼명 내 줄바꿈(\n), 공백, 전각 문자 완벽 제거
df.columns = [str(c).replace('\n', '').replace(' ', '').strip() for c in df.columns]

# ---------------------------------------------------------
# 2. 필수 컬럼 유연 감지 (KeyError 완전 방지)
# ---------------------------------------------------------
# 지정일시_FROM/TO 탐색
from_col = next((c for c in df.columns if '지정일시' in c and 'FROM' in c.upper()), None)
to_col = next((c for c in df.columns if '지정일시' in c and 'TO' in c.upper()), None)
app_col = next((c for c in df.columns if '신청일시' in c or '요청일시' in c), None)
purpose_col = next((c for c in df.columns if '사용목적' in c), '사용목적명')

print(f"감지된 지정일시_FROM : {from_col}")
print(f"감지된 지정일시_TO   : {to_col}")
print(f"감지된 신청일시       : {app_col}")

# 날짜형 Datetime 변환
date_cols = [from_col, to_col, app_col, '입항일시', '출항일시']
for col in date_cols:
    if col and col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# ---------------------------------------------------------
# 3. 대기시간(TW) 계산 (단위: 시간)
# ---------------------------------------------------------
# 지정일시_TO - 지정일시_FROM (블로그 방식)
if from_col and to_col and from_col in df.columns and to_col in df.columns:
    df['대기시간_TW'] = (df[to_col] - df[from_col]).dt.total_seconds() / 3600.0
else:
    df['대기시간_TW'] = 0.0

# 음수값 0 보정 및 결측치 0 처리
df['대기시간_TW'] = df['대기시간_TW'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0.0)

# ---------------------------------------------------------
# 4. 접안/출항 대기시점 추출 (선박별 그룹화)
# ---------------------------------------------------------
key_cols = [c for c in ['호출부호', '선박명', '입항횟수_연도', '입항횟수_년도', '입항횟수_횟수'] if c in df.columns]

# 접안 관련 신청 건의 최종시점
berth_df = df[df[purpose_col].astype(str).str.contains('접안|급유|계선|화물', na=False)]
if not berth_df.empty and from_col in df.columns:
    berth_wait = berth_df.groupby(key_cols)[from_col].max().reset_index()
    berth_wait.rename(columns={from_col: '최종접안대기시점'}, inplace=True)
    df = pd.merge(df, berth_wait, on=key_cols, how='left')
else:
    df['최종접안대기시점'] = pd.NaT

# 출항 관련 신청 건의 최초시점
dept_df = df[df[purpose_col].astype(str).str.contains('출항', na=False)]
if not dept_df.empty and from_col in df.columns:
    dept_wait = dept_df.groupby(key_cols)[from_col].min().reset_index()
    dept_wait.rename(columns={from_col: '최초출항대기시점'}, inplace=True)
    df = pd.merge(df, dept_wait, on=key_cols, how='left')
else:
    df['최초출항대기시점'] = pd.NaT

# ---------------------------------------------------------
# 5. 블로그 4가지 Case별 서비스시간(TS) 및 대기율(WR) 산출
# ---------------------------------------------------------
cond1 = df['최종접안대기시점'].isna() & df['최초출항대기시점'].isna()
cond2 = df['최종접안대기시점'].isna() & df['최초출항대기시점'].notna()
cond3 = df['최종접안대기시점'].notna() & df['최초출항대기시점'].isna()
cond4 = df['최종접안대기시점'].notna() & df['최초출항대기시점'].notna()

df['서비스시간_TS'] = np.select(
    [cond1, cond2, cond3, cond4],
    [
        (df['출항일시'] - df['입항일시']).dt.total_seconds() / 3600.0,
        (df['최초출항대기시점'] - df['입항일시']).dt.total_seconds() / 3600.0,
        (df['출항일시'] - df['최종접안대기시점']).dt.total_seconds() / 3600.0,
        (df['최초출항대기시점'] - df['최종접안대기시점']).dt.total_seconds() / 3600.0
    ],
    default=np.nan
)

# 서비스시간 이상치(0 이하 및 결측치) 보정
df['서비스시간_TS'] = df['서비스시간_TS'].apply(lambda x: x if pd.notnull(x) and x > 0 else np.nan)

# 대기율 (WR = TW / TS)
df['대기율_WR'] = df['대기시간_TW'] / df['서비스시간_TS']

print("\n=== [PORT MIS 3.3 대기시간 / 서비스시간 / 대기율 산출 결과] ===")
print(df[['호출부호', '선박명', '사용목적명', '대기시간_TW', '서비스시간_TS', '대기율_WR']].head(10))

감지된 지정일시_FROM : None
감지된 지정일시_TO   : None
감지된 신청일시       : 신청일시

=== [PORT MIS 3.3 대기시간 / 서비스시간 / 대기율 산출 결과] ===
     호출부호      선박명 사용목적명  대기시간_TW       서비스시간_TS  대기율_WR
0  796256      백경3    기타      0.0  182432.000000     0.0
1  058251      대은호    계선      0.0   73423.733333     0.0
2  058251      대은호    계선      0.0   73423.733333     0.0
3  058251      대은호    기타      0.0   73423.733333     0.0
4  915291    17금영호    기타      0.0   46038.000000     0.0
5   JVNF8  SUNRISE    수리      0.0   39161.500000     0.0
6  030713      청운호   NaN      0.0   32832.000000     0.0
7   DSDN4     성진1호    계선      0.0   29134.750000     0.0
8  930380   102진양호    기타      0.0   22502.533333     0.0
9  930380   102진양호    기타      0.0   22502.533333     0.0


In [56]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. merged_df.csv 불러오기
# ---------------------------------------------------------
file_path = 'processed_data/merged_df.csv'  # 파일 경로에 맞춰 수정해 주세요
df = pd.read_csv(file_path)

print("데이터 로드 완료! 전체 크기:", df.shape)

# ---------------------------------------------------------
# 2. Key 컬럼 이름 및 데이터 타입 표준화
# ---------------------------------------------------------
# '년도' 컬럼이 존재할 경우 '연도'로 표준화
if '입항횟수_년도' in df.columns:
  df.rename(columns={'입항횟수_년도': '입항횟수_연도'}, inplace=True)

key_cols = ['호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수']

# Key 컬럼 데이터 타입을 문자열(str)로 정제하여 그룹화 및 조인 오류 방지
for k in key_cols:
  if k in df.columns:
    df[k] = df[k].astype(str).str.replace('.0', '', regex=False).str.strip()

# ---------------------------------------------------------
# 3. 날짜/시간 컬럼 Datetime 형변환
# ---------------------------------------------------------
datetime_cols = ['지정일시_FROM', '지정일시_TO', '신청일시', '입항일시', '출항일시']
for col in datetime_cols:
  if col in df.columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# ---------------------------------------------------------
# 4. [TW] 개별 행 단위 대기시간(시간 단위) 산출
# ---------------------------------------------------------
df['접안대기_여부'] = (
    df['사용목적명'].astype(str).str.contains('접안대기', na=False)
)
df['출항대기_여부'] = (
    df['사용목적명'].astype(str).str.contains('출항대기', na=False)
)

df['대기시간_TW'] = np.where(
    df['접안대기_여부'],
    (df['지정일시_TO'] - df['지정일시_FROM']).dt.total_seconds() / 3600.0,
    0.0,
)
# 음수값 방지 (0 이하 값은 0.0 처리)
df['대기시간_TW'] = df['대기시간_TW'].apply(lambda x: max(x, 0.0))

# ---------------------------------------------------------
# 5. 입항 건별 '최종 접안대기 시점' 및 '최초 출항대기 시점' 추출
# ---------------------------------------------------------
berth_df = df[df['접안대기_여부']]
if not berth_df.empty:
  berth_wait = berth_df.groupby(key_cols)['지정일시_TO'].max().reset_index()
  berth_wait.rename(columns={'지정일시_TO': '최종접안대기시점'}, inplace=True)
  df = pd.merge(df, berth_wait, on=key_cols, how='left')
else:
  df['최종접안대기시점'] = pd.NaT

dept_df = df[df['출항대기_여부']]
if not dept_df.empty:
  dept_wait = dept_df.groupby(key_cols)['지정일시_FROM'].min().reset_index()
  dept_wait.rename(columns={'지정일시_FROM': '최초출항대기시점'}, inplace=True)
  df = pd.merge(df, dept_wait, on=key_cols, how='left')
else:
  df['최초출항대기시점'] = pd.NaT

# ---------------------------------------------------------
# 6. [TS] 서비스시간(4가지 조건 분기) 및 [WR] 대기율 산출
# ---------------------------------------------------------
cond1 = df['최종접안대기시점'].isna() & df['최초출항대기시점'].isna()
cond2 = df['최종접안대기시점'].isna() & df['최초출항대기시점'].notna()
cond3 = df['최종접안대기시점'].notna() & df['최초출항대기시점'].isna()
cond4 = df['최종접안대기시점'].notna() & df['최초출항대기시점'].notna()

df['서비스시간_TS'] = np.select(
    [cond1, cond2, cond3, cond4],
    [
        (df['출항일시'] - df['입항일시']).dt.total_seconds() / 3600.0,
        (df['최초출항대기시점'] - df['입항일시']).dt.total_seconds() / 3600.0,
        (df['출항일시'] - df['최종접안대기시점']).dt.total_seconds() / 3600.0,
        (
            df['최초출항대기시점'] - df['최종접안대기시점']
        ).dt.total_seconds()
        / 3600.0,
    ],
    default=np.nan,
)

# 대기율 (0으로 나누기 예방 처리)
df['대기율_WR'] = np.where(
    (df['서비스시간_TS'].notna()) & (df['서비스시간_TS'] > 0),
    df['대기시간_TW'] / df['서비스시간_TS'],
    0.0,
)

# ---------------------------------------------------------
# 7. 결과 저장 및 통계 출력
# ---------------------------------------------------------
output_path = 'processed_data/calculated_merged_df.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'\n 성공적으로 연산 완료되어 {output_path} 에 저장되었습니다.')

# 접안대기 건수 및 대기시간 통계 확인
berth_count = df['접안대기_여부'].sum()
print(f" 전체 {len(df)}개 행 중 '접안대기' 건수: {berth_count}건")
print(' 대기시간_TW 평균(전체):', round(df['대기시간_TW'].mean(), 2), '시간')
print(
    ' 대기시간_TW 평균(접안대기건만):',
    round(df[df['접안대기_여부']]['대기시간_TW'].mean(), 2),
    '시간',
)

데이터 로드 완료! 전체 크기: (85483, 23)

 성공적으로 연산 완료되어 processed_data/calculated_merged_df.csv 에 저장되었습니다.
 전체 85483개 행 중 '접안대기' 건수: 14437건
 대기시간_TW 평균(전체): 105.76 시간
 대기시간_TW 평균(접안대기건만): 626.24 시간


In [57]:
import numpy as np
import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv('processed_data/merged_df.csv')

# 컬럼명 및 키 정제
if '입항횟수_년도' in df.columns:
  df.rename(columns={'입항횟수_년도': '입항횟수_연도'}, inplace=True)

key_cols = ['호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수']

for k in key_cols:
  df[k] = df[k].astype(str).str.replace('.0', '', regex=False).str.strip()

# 날짜/시간 변환
datetime_cols = ['지정일시_FROM', '지정일시_TO', '신청일시', '입항일시', '출항일시']
for col in datetime_cols:
  df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. 접안대기 / 출항대기 구분 및 대기시간(TW) 계산
df['접안대기_여부'] = (
    df['사용목적명'].astype(str).str.contains('접안대기', na=False)
)
df['출항대기_여부'] = (
    df['사용목적명'].astype(str).str.contains('출항대기', na=False)
)

df['대기시간_TW'] = np.where(
    df['접안대기_여부'],
    (df['지정일시_TO'] - df['지정일시_FROM']).dt.total_seconds() / 3600.0,
    0.0,
)
df['대기시간_TW'] = df['대기시간_TW'].apply(lambda x: max(x, 0.0))

# 3. 입항 건별 시점 추출 (최종접안대기시점, 최초출항대기시점)
berth_times = (
    df[df['접안대기_여부']]
    .groupby(key_cols)['지정일시_TO']
    .max()
    .reset_index()
    .rename(columns={'지정일시_TO': '최종접안대기시점'})
)

dept_times = (
    df[df['출항대기_여부']]
    .groupby(key_cols)['지정일시_FROM']
    .min()
    .reset_index()
    .rename(columns={'지정일시_FROM': '최초출항대기시점'})
)

# 기본 입항/출항일시 및 대기시간 합계 그룹화
summary_df = (
    df.groupby(key_cols)
    .agg(
        {
            '입항일시': 'min',
            '출항일시': 'max',
            '대기시간_TW': 'sum',  # 입항 건당 총 대기시간 합산
        }
    )
    .reset_index()
)

# 시점 병합
summary_df = pd.merge(summary_df, berth_times, on=key_cols, how='left')
summary_df = pd.merge(summary_df, dept_times, on=key_cols, how='left')

# 4. 서비스시간(TS) 조건별 계산
cond1 = (
    summary_df['최종접안대기시점'].isna() & summary_df['최초출항대기시점'].isna()
)
cond2 = (
    summary_df['최종접안대기시점'].isna() & summary_df['최초출항대기시점'].notna()
)
cond3 = (
    summary_df['최종접안대기시점'].notna() & summary_df['최초출항대기시점'].isna()
)
cond4 = (
    summary_df['최종접안대기시점'].notna() & summary_df['최초출항대기시점'].notna()
)

summary_df['서비스시간_TS'] = np.select(
    [cond1, cond2, cond3, cond4],
    [
        (summary_df['출항일시'] - summary_df['입항일시']).dt.total_seconds()
        / 3600.0,
        (
            summary_df['최초출항대기시점'] - summary_df['입항일시']
        ).dt.total_seconds()
        / 3600.0,
        (summary_df['출항일시'] - summary_df['최종접안대기시점']).dt.total_seconds()
        / 3600.0,
        (
            summary_df['최초출항대기시점'] - summary_df['최종접안대기시점']
        ).dt.total_seconds()
        / 3600.0,
    ],
    default=np.nan,
)

# 5. 대기율(WR) 계산
summary_df['대기율_WR'] = np.where(
    (summary_df['서비스시간_TS'].notna()) & (summary_df['서비스시간_TS'] > 0),
    summary_df['대기시간_TW'] / summary_df['서비스시간_TS'],
    0.0,
)

# 결과 저장
summary_df.to_csv(
    'processed_data/port_waiting_summary.csv', index=False, encoding='utf-8-sig'
)
print('입항 건별 대기율 계산 완료!')

입항 건별 대기율 계산 완료!


In [58]:
# ---------------------------------------------------------
# [추가] 전체 평균 대기시간, 서비스시간, 대기율 출력 코드
# ---------------------------------------------------------
print("=" * 50)
print("📊 전체 입항 건 기준 주요 지표 평균")
print("=" * 50)

# 1. 평균 대기시간 (TW)
avg_tw_all = summary_df['대기시간_TW'].mean()
avg_tw_waited = summary_df[summary_df['대기시간_TW'] > 0]['대기시간_TW'].mean()

# 2. 평균 서비스시간 (TS)
avg_ts = summary_df['서비스시간_TS'].mean()

# 3. 평균 대기율 (WR)
avg_wr_all = summary_df['대기율_WR'].mean()
avg_wr_waited = summary_df[summary_df['대기시간_TW'] > 0]['대기율_WR'].mean()

print(f"1. 평균 대기시간(TW): {avg_tw_all:.2f} 시간 (전체 입항 건 기준)")
print(f"   └ 대기 발생 건만 기준: {avg_tw_waited:.2f} 시간")
print(f"2. 평균 서비스시간(TS): {avg_ts:.2f} 시간")
print(
    f"3. 평균 대기율(WR): {avg_wr_all * 100:.2f}% ({avg_wr_all:.4f}) (전체"
    " 기준)"
)
print(f"   └ 대기 발생 건만 기준: {avg_wr_waited * 100:.2f}% ({avg_wr_waited:.4f})")
print("=" * 50)

📊 전체 입항 건 기준 주요 지표 평균
1. 평균 대기시간(TW): 200.29 시간 (전체 입항 건 기준)
   └ 대기 발생 건만 기준: 1315.25 시간
2. 평균 서비스시간(TS): -115.76 시간
3. 평균 대기율(WR): 235.18% (2.3518) (전체 기준)
   └ 대기 발생 건만 기준: 1544.34% (15.4434)


In [59]:
# views/port1.py
import pandas as pd
import streamlit as st


def port_1():
  st.title('⚓ 북항 운영 대시보드')
  st.caption('PORT-MIS 대기시간, 서비스시간 및 대기율(WR) 모니터링')

  # 1. 데이터 로드 (캐싱을 통해 속도 최적화)
  @st.cache_data
  def load_data():
    df = pd.read_csv('processed_data/port_waiting_summary.csv')
    # 북항 데이터만 필터링 (데이터에 항구 구분 컬럼이 있는 경우 적용)
    # df = df[df['항명'] == '부산']
    return df

  try:
    df = load_data()
  except FileNotFoundError:
    st.error(
      '데이터 파일을 찾을 수 없습니다. `processed_data/port_waiting_summary.csv` 경로를 확인해주세요.'
    )
    return

  # 2. 주요 KPI 카드 (Metric)
  st.subheader('📌 주요 지표 요약')
  col1, col2, col3, col4 = st.columns(4)

  avg_tw = df['대기시간_TW'].mean()
  avg_ts = df['서비스시간_TS'].mean()
  avg_wr = df['대기율_WR'].mean()
  total_ships = len(df)

  with col1:
    st.metric(label='총 입항 건수', value=f'{total_ships:,} 건')

  with col2:
    st.metric(label='평균 대기시간 (TW)', value=f'{avg_tw:.2f} 시간')

  with col3:
    st.metric(label='평균 서비스시간 (TS)', value=f'{avg_ts:.2f} 시간')

  with col4:
    st.metric(label='평균 대기율 (WR)', value=f'{avg_wr * 100:.2f} %')

  st.divider()

  # 3. 상세 분석 및 차트
  col_left, col_right = st.columns([1, 1])

  with col_left:
    st.subheader('📊 대기시간(TW) 분포')
    # 대기시간 0 초과 건만 필터링하여 히스토그램 표시
    tw_waited = df[df['대기시간_TW'] > 0]['대기시간_TW']
    st.bar_chart(tw_waited.value_counts(bins=10))

  with col_right:
    st.subheader('📈 상위 대기 선박 Top 5')
    top_waiting = (
      df.sort_values(by='대기시간_TW', ascending=False)
      .head(5)[['선박명', '대기시간_TW', '서비스시간_TS', '대기율_WR']]
      .reset_index(drop=True)
    )
    st.dataframe(top_waiting, use_container_width=True)

  # 4. 전체 데이터 테이블
  st.subheader('📑 상세 데이터 조회')
  st.dataframe(df, use_container_width=True)